In [5]:
# read parquet as df
import pandas as pd

df = pd.read_parquet("test_pairs_with_siglip_embeddings.parquet")

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import normalize

# ---- CONFIG ----
fraud_prefix = "fraud_emb_"
real_prefix  = "real_emb_"
label_col    = "label"

# ---- 1. Extract embedding columns ----
fraud_cols = sorted([c for c in df.columns if c.startswith(fraud_prefix)])
real_cols  = sorted([c for c in df.columns if c.startswith(real_prefix)])

assert len(fraud_cols) > 0, "No fraud embedding columns found."
assert len(real_cols)  > 0, "No real embedding columns found."
assert len(fraud_cols) == len(real_cols), "Embedding dimensions mismatch."

# ---- 2. Convert to numpy arrays ----
fraud_emb = df[fraud_cols].values.astype(np.float32)
real_emb  = df[real_cols].values.astype(np.float32)

# ---- 3. L2 normalize (important for cosine similarity) ----
fraud_emb = normalize(fraud_emb, axis=1)
real_emb  = normalize(real_emb, axis=1)

# ---- 4. Cosine similarity ----
cos_sim = np.sum(fraud_emb * real_emb, axis=1)

# ---- 5. ROC AUC ----
y_true = df[label_col].values
roc_auc = roc_auc_score(y_true, cos_sim)

print(f"ROC AUC (cosine similarity): {roc_auc:.6f}")

ROC AUC (cosine similarity): 0.778219


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

# ----------------------------
# CONFIG
# ----------------------------
pt_path = "single_run_model.pt"  # or single_run_model.pt
label_col = "label"

fraud_prefix = "fraud_emb_"
real_prefix  = "real_emb_"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

# ----------------------------
# LOAD CHECKPOINT / STATE_DICT (robust)
# ----------------------------
obj = torch.load(pt_path, map_location="cpu")

if isinstance(obj, dict) and "model_state" in obj:
    state_dict = obj["model_state"]
else:
    state_dict = obj  # raw state_dict

# Must contain these keys
for k in ["head.0.weight", "head.0.bias", "head.2.weight", "head.2.bias"]:
    if k not in state_dict:
        raise KeyError(f"Expected key '{k}' not found. Keys present (sample): {list(state_dict.keys())[:20]}")

w0 = state_dict["head.0.weight"]
w2 = state_dict["head.2.weight"]

hidden_dim = int(w0.shape[0])
in_dim     = int(w0.shape[1])
out_dim    = int(w2.shape[0])

print(f"[INFO] Inferred dims from checkpoint: in_dim={in_dim}, hidden_dim={hidden_dim}, out_dim={out_dim}")

# ----------------------------
# DEFINE MODEL THAT MATCHES STATE_DICT
# ----------------------------
class SiameseEmbeddingModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, out_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x1, x2):
        z1 = self.head(x1)
        z2 = self.head(x2)
        return z1, z2

model = SiameseEmbeddingModel(in_dim, hidden_dim, out_dim).to(device)
model.load_state_dict(state_dict, strict=True)
model.eval()
print("[INFO] Model loaded.")

# ----------------------------
# HELPERS
# ----------------------------
def _to_float32_matrix(df_sub: pd.DataFrame) -> np.ndarray:
    return df_sub.to_numpy(dtype=np.float32, copy=False)

@torch.no_grad()
def roc_auc_from_two_mats(mat1: np.ndarray, mat2: np.ndarray, y_true: np.ndarray, batch_size: int = 8192) -> float:
    assert mat1.shape == mat2.shape
    n = mat1.shape[0]

    sims_all = []
    for start in range(0, n, batch_size):
        end = min(n, start + batch_size)
        x1 = torch.from_numpy(mat1[start:end]).to(device)
        x2 = torch.from_numpy(mat2[start:end]).to(device)

        z1, z2 = model(x1, x2)
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        sims = F.cosine_similarity(z1, z2, dim=1)
        sims_all.append(sims.detach().cpu())

    sims = torch.cat(sims_all, dim=0).numpy()
    return float(roc_auc_score(y_true, sims))

@torch.no_grad()
def roc_auc_raw_cosine(mat1: np.ndarray, mat2: np.ndarray) -> np.ndarray:
    # raw cosine on inputs (for sanity)
    x1 = F.normalize(torch.from_numpy(mat1), dim=1)
    x2 = F.normalize(torch.from_numpy(mat2), dim=1)
    return F.cosine_similarity(x1, x2, dim=1).numpy()

# ----------------------------
# LABELS
# ----------------------------
y = df[label_col].to_numpy()
y = y.astype(np.int32, copy=False)

# ----------------------------
# (A) PREFIX-BASED: fraud_txt_emb_*, real_txt_emb_*
# ----------------------------
fraud_cols = sorted([c for c in df.columns if c.startswith(fraud_prefix)])
real_cols  = sorted([c for c in df.columns if c.startswith(real_prefix)])

print(f"[INFO] Prefix cols: fraud={len(fraud_cols)}, real={len(real_cols)}")

if len(fraud_cols) == in_dim and len(real_cols) == in_dim:
    fraud_mat = _to_float32_matrix(df[fraud_cols])
    real_mat  = _to_float32_matrix(df[real_cols])

    auc_proj_prefix = roc_auc_from_two_mats(fraud_mat, real_mat, y)
    auc_raw_prefix  = roc_auc_score(y, roc_auc_raw_cosine(fraud_mat, real_mat))

    print(f"ROC AUC (RAW cosine, prefix inputs):   {auc_raw_prefix:.6f}")
    print(f"ROC AUC (MODEL cosine, prefix inputs): {auc_proj_prefix:.6f}")
else:
    print(f"[WARN] Prefix-based eval skipped because prefix dims != in_dim ({in_dim}).")

# ----------------------------
# (B) EVALUATOR-STYLE SLICES (generalized to in_dim)
#     This matches your Evaluator approach if your parquet is laid out like:
#     [names..., label, then x1, then x2]
# ----------------------------
start_x1 = 3
start_x2 = start_x1 + in_dim
end_x2   = start_x2 + in_dim

if df.shape[1] >= end_x2:
    x1_slice = df.iloc[:, start_x1:start_x2]
    x2_slice = df.iloc[:, start_x2:end_x2]

    x1_mat = _to_float32_matrix(x1_slice)
    x2_mat = _to_float32_matrix(x2_slice)

    auc_proj_slice = roc_auc_from_two_mats(x1_mat, x2_mat, y)
    auc_raw_slice  = roc_auc_score(y, roc_auc_raw_cosine(x1_mat, x2_mat))

    print(f"ROC AUC (RAW cosine, slice inputs):    {auc_raw_slice:.6f}")
    print(f"ROC AUC (MODEL cosine, slice inputs):  {auc_proj_slice:.6f}")
else:
    print(f"[WARN] Slice-based eval skipped: df has {df.shape[1]} cols, need at least {end_x2}.")

[INFO] Inferred dims from checkpoint: in_dim=768, hidden_dim=768, out_dim=768
[INFO] Model loaded.
[INFO] Prefix cols: fraud=768, real=768
ROC AUC (RAW cosine, prefix inputs):   0.927718
ROC AUC (MODEL cosine, prefix inputs): 0.898852
ROC AUC (RAW cosine, slice inputs):    0.927718
ROC AUC (MODEL cosine, slice inputs):  0.988039


In [13]:
import os
import glob
import numpy as np
import pandas as pd

DIR = ""
SUFFIX = "_validate_pairs_ref_10k.parquet"

def youden_threshold(scores: np.ndarray, y: np.ndarray) -> float:
    """
    Choose threshold t maximizing Youden's J = TPR + TNR - 1,
    using rule: predict positive iff score >= t.
    Assumes y in {0,1} and higher score => more likely y==1.
    """
    scores = np.asarray(scores, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)

    if scores.shape[0] != y.shape[0]:
        raise ValueError("scores and y must have same length")
    if scores.shape[0] == 0:
        raise ValueError("empty input")
    if not np.isin(y, [0, 1]).all():
        raise ValueError("y must be binary {0,1}")

    P = int(y.sum())
    N = int((1 - y).sum())
    if P == 0 or N == 0:
        # Degenerate: no meaningful threshold
        return float(np.median(scores))

    # Sort by score descending
    order = np.argsort(scores)[::-1]
    s = scores[order]
    yy = y[order]

    # cumulative TP when predicting positive for top k
    tp_cum = np.cumsum(yy)

    # group ends for each unique score (so threshold at that score includes all ties)
    ends = np.r_[np.where(np.diff(s) != 0)[0], len(s) - 1]

    best_J = -np.inf
    best_t = float(s[ends[0]])

    for e in ends:
        tp = int(tp_cum[e])
        fp = int((e + 1) - tp)
        fn = P - tp
        tn = N - fp

        tpr = tp / P
        tnr = tn / N
        J = tpr + tnr - 1.0  # == tpr - fpr

        if J > best_J:
            best_J = J
            best_t = float(s[e])

    return best_t


# ---- find all font parquets ----
paths = sorted(glob.glob(os.path.join(DIR, f"*{SUFFIX}")))
if not paths:
    raise FileNotFoundError(f"No parquets found at {DIR}/*{SUFFIX}")

thresholds = {}
for p in paths:
    font = os.path.basename(p).replace(SUFFIX, "")
    df = pd.read_parquet(p)

    if "label" not in df.columns or "cosine_sim" not in df.columns:
        raise ValueError(f"{p} missing required columns. Has: {list(df.columns)}")

    y = df["label"].to_numpy()
    s = df["cosine_sim"].to_numpy()

    t = youden_threshold(s, y)
    thresholds[font] = t

# ---- print pasteable dict (sorted) ----
print("font_thresholds = {")
for font in sorted(thresholds.keys()):
    print(f'    "{font}": {thresholds[font]:.16f},')
print("}")

font_thresholds = {
    "arimo": 0.8132318854331970,
    "charissil": 0.8223531246185303,
    "cousine": 0.8044030666351318,
    "dejavusans": 0.8149772286415100,
    "doulossil": 0.8242787122726440,
    "exo2": 0.8193700313568115,
    "freeserif": 0.8264714479446411,
    "gentiumplus": 0.8164674639701843,
    "librefranklin": 0.7957530021667480,
    "notosans": 0.8146908283233643,
    "unifont": 0.8102402091026306,
    "vollkorn": 0.8139773607254028,
}


In [ ]:
import pandas as pd

print(df.head())

  fraudulent_name    real_name  label  cosine_sim
0      meĝaĉriŧic   megacritic    1.0    0.990189
1         thedodo    queveohoy    0.0    0.645260
2       alrnundoz      almundo    1.0    0.848608
3          nijobs      topjobs    0.0    0.839926
4     trhonethrow  thronethrow    1.0    0.985071


In [ ]:
df = pd.read_parquet("dejavusans_validate_pairs_ref_10k.parquet")
print(df.head())

  fraudulent_name    real_name  label  fraud_emb_0  fraud_emb_1  fraud_emb_2  \
0      meĝaĉriŧic   megacritic    1.0     0.023132    -0.020416     0.022003   
1         thedodo    queveohoy    0.0    -0.009911    -0.015358    -0.029938   
2       alrnundoz      almundo    1.0    -0.031616    -0.026779     0.020187   
3          nijobs      topjobs    0.0     0.000953    -0.019272    -0.002138   
4     trhonethrow  thronethrow    1.0     0.031982    -0.009552    -0.018539   

   fraud_emb_3  fraud_emb_4  fraud_emb_5  fraud_emb_6  ...  real_emb_758  \
0     0.008713    -0.048828     0.026901    -0.025223  ...     -0.003302   
1     0.023041    -0.026993     0.035919    -0.026840  ...     -0.012299   
2    -0.028610    -0.045776     0.039490     0.058868  ...     -0.025299   
3     0.034210    -0.023788    -0.015221     0.015823  ...     -0.023865   
4     0.005241    -0.030182     0.022903     0.014000  ...     -0.019867   

   real_emb_759  real_emb_760  real_emb_761  real_emb_762  rea

In [11]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

# ----------------------------
# CONFIG
# ----------------------------
pt_path = "single_run_model.pt"  # or single_run_model.pt
label_col = "label"

fraud_prefix = "fraud_emb_"
real_prefix  = "real_emb_"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

# ----------------------------
# LOAD CHECKPOINT / STATE_DICT (robust)
# ----------------------------
obj = torch.load(pt_path, map_location="cpu")

if isinstance(obj, dict) and "model_state" in obj:
    state_dict = obj["model_state"]
else:
    state_dict = obj  # raw state_dict

# Must contain these keys
for k in ["head.0.weight", "head.0.bias", "head.2.weight", "head.2.bias"]:
    if k not in state_dict:
        raise KeyError(f"Expected key '{k}' not found. Keys present (sample): {list(state_dict.keys())[:20]}")

w0 = state_dict["head.0.weight"]
w2 = state_dict["head.2.weight"]

hidden_dim = int(w0.shape[0])
in_dim     = int(w0.shape[1])
out_dim    = int(w2.shape[0])

print(f"[INFO] Inferred dims from checkpoint: in_dim={in_dim}, hidden_dim={hidden_dim}, out_dim={out_dim}")

# ----------------------------
# DEFINE MODEL THAT MATCHES STATE_DICT
# ----------------------------
class SiameseEmbeddingModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, out_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x1, x2):
        z1 = self.head(x1)
        z2 = self.head(x2)
        return z1, z2

model = SiameseEmbeddingModel(in_dim, hidden_dim, out_dim).to(device)
model.load_state_dict(state_dict, strict=True)
model.eval()
print("[INFO] Model loaded.")

# ----------------------------
# HELPERS
# ----------------------------
def _to_float32_matrix(df_sub: pd.DataFrame) -> np.ndarray:
    return df_sub.to_numpy(dtype=np.float32, copy=False)

@torch.no_grad()
def roc_auc_from_two_mats(mat1: np.ndarray, mat2: np.ndarray, y_true: np.ndarray, batch_size: int = 8192) -> float:
    assert mat1.shape == mat2.shape
    n = mat1.shape[0]

    sims_all = []
    for start in range(0, n, batch_size):
        end = min(n, start + batch_size)
        x1 = torch.from_numpy(mat1[start:end]).to(device)
        x2 = torch.from_numpy(mat2[start:end]).to(device)

        z1, z2 = model(x1, x2)
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        sims = F.cosine_similarity(z1, z2, dim=1)
        sims_all.append(sims.detach().cpu())

    sims = torch.cat(sims_all, dim=0).numpy()
    return float(roc_auc_score(y_true, sims))

@torch.no_grad()
def roc_auc_raw_cosine(mat1: np.ndarray, mat2: np.ndarray) -> np.ndarray:
    # raw cosine on inputs (for sanity)
    x1 = F.normalize(torch.from_numpy(mat1), dim=1)
    x2 = F.normalize(torch.from_numpy(mat2), dim=1)
    return F.cosine_similarity(x1, x2, dim=1).numpy()

# ----------------------------
# LABELS
# ----------------------------
y = df[label_col].to_numpy()
y = y.astype(np.int32, copy=False)

# ----------------------------
# (A) PREFIX-BASED: fraud_txt_emb_*, real_txt_emb_*
# ----------------------------
fraud_cols = sorted([c for c in df.columns if c.startswith(fraud_prefix)])
real_cols  = sorted([c for c in df.columns if c.startswith(real_prefix)])

print(f"[INFO] Prefix cols: fraud={len(fraud_cols)}, real={len(real_cols)}")

if len(fraud_cols) == in_dim and len(real_cols) == in_dim:
    fraud_mat = _to_float32_matrix(df[fraud_cols])
    real_mat  = _to_float32_matrix(df[real_cols])

    auc_proj_prefix = roc_auc_from_two_mats(fraud_mat, real_mat, y)
    auc_raw_prefix  = roc_auc_score(y, roc_auc_raw_cosine(fraud_mat, real_mat))

    print(f"ROC AUC (RAW cosine, prefix inputs):   {auc_raw_prefix:.6f}")
    print(f"ROC AUC (MODEL cosine, prefix inputs): {auc_proj_prefix:.6f}")
else:
    print(f"[WARN] Prefix-based eval skipped because prefix dims != in_dim ({in_dim}).")

# ----------------------------
# (B) EVALUATOR-STYLE SLICES (generalized to in_dim)
#     This matches your Evaluator approach if your parquet is laid out like:
#     [names..., label, then x1, then x2]
# ----------------------------
start_x1 = 3
start_x2 = start_x1 + in_dim
end_x2   = start_x2 + in_dim

if df.shape[1] >= end_x2:
    x1_slice = df.iloc[:, start_x1:start_x2]
    x2_slice = df.iloc[:, start_x2:end_x2]

    x1_mat = _to_float32_matrix(x1_slice)
    x2_mat = _to_float32_matrix(x2_slice)

    auc_proj_slice = roc_auc_from_two_mats(x1_mat, x2_mat, y)
    auc_raw_slice  = roc_auc_score(y, roc_auc_raw_cosine(x1_mat, x2_mat))

    print(f"ROC AUC (RAW cosine, slice inputs):    {auc_raw_slice:.6f}")
    print(f"ROC AUC (MODEL cosine, slice inputs):  {auc_proj_slice:.6f}")
else:
    print(f"[WARN] Slice-based eval skipped: df has {df.shape[1]} cols, need at least {end_x2}.")

[INFO] Inferred dims from checkpoint: in_dim=768, hidden_dim=768, out_dim=768
[INFO] Model loaded.
[INFO] Prefix cols: fraud=0, real=0
[WARN] Prefix-based eval skipped because prefix dims != in_dim (768).
[WARN] Slice-based eval skipped: df has 4 cols, need at least 1539.


In [25]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================
# CONFIG
# ============================

pt_path = "single_run_model.pt"
input_data_path = "validate_pairs_with_siglip_embeddings.parquet"   # change if needed
output_path = "projected_embeddings.parquet"

label_col = "label"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

# ============================
# LOAD DATAFRAME
# ============================

df = pd.read_parquet(input_data_path)

print("Loaded dataframe shape:", df.shape)


# ============================
# LOAD CHECKPOINT
# ============================

obj = torch.load(pt_path, map_location="cpu")

if isinstance(obj, dict) and "model_state" in obj:
    state_dict = obj["model_state"]
else:
    state_dict = obj


# infer dimensions from checkpoint
w0 = state_dict["head.0.weight"]
w2 = state_dict["head.2.weight"]

hidden_dim = int(w0.shape[0])
in_dim = int(w0.shape[1])
out_dim = int(w2.shape[0])

print("Model dims:")
print("in_dim:", in_dim)
print("hidden_dim:", hidden_dim)
print("out_dim:", out_dim)


# ============================
# MODEL DEFINITION
# ============================

class SiameseEmbeddingModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, out_dim):
        super().__init__()

        self.head = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x1, x2):

        z1 = self.head(x1)
        z2 = self.head(x2)

        return z1, z2


model = SiameseEmbeddingModel(in_dim, hidden_dim, out_dim).to(device)
model.load_state_dict(state_dict)
model.eval()

print("Model loaded.")


# ============================
# EXTRACT EMBEDDING SLICES
# ============================

start_x1 = 3
start_x2 = start_x1 + in_dim
end_x2 = start_x2 + in_dim

x1_slice = df.iloc[:, start_x1:start_x2]
x2_slice = df.iloc[:, start_x2:end_x2]

x1_mat = x1_slice.to_numpy(dtype=np.float32)
x2_mat = x2_slice.to_numpy(dtype=np.float32)

print("Embedding matrices:")
print("x1:", x1_mat.shape)
print("x2:", x2_mat.shape)


# ============================
# PROJECT EMBEDDINGS THROUGH MODEL
# ============================

@torch.no_grad()
def compute_projected_embeddings(mat1, mat2, batch_size=8192):

    n = mat1.shape[0]

    z1_all = []
    z2_all = []

    for start in range(0, n, batch_size):

        end = min(start + batch_size, n)

        x1 = torch.from_numpy(mat1[start:end]).to(device)
        x2 = torch.from_numpy(mat2[start:end]).to(device)

        z1, z2 = model(x1, x2)

        z1_all.append(z1.cpu())
        z2_all.append(z2.cpu())

    z1 = torch.cat(z1_all).numpy()
    z2 = torch.cat(z2_all).numpy()

    return z1, z2


z1, z2 = compute_projected_embeddings(x1_mat, x2_mat)

print("Projected embeddings:")
print("z1:", z1.shape)
print("z2:", z2.shape)


# ============================
# BUILD OUTPUT DATAFRAME
# ============================

z1_df = pd.DataFrame(z1).add_prefix("fraud_emb_")
z2_df = pd.DataFrame(z2).add_prefix("real_emb_")

proj_df = pd.concat(
    [
        df[["fraudulent_name", "real_name", label_col]],
        z1_df,
        z2_df
    ],
    axis=1
)

print("Output dataframe shape:", proj_df.shape)


# ============================
# SAVE
# ============================

proj_df.to_parquet(output_path, index=False)

print("Saved projected embeddings to:", output_path)

Device: mps
Loaded dataframe shape: (9999, 1539)
Model dims:
in_dim: 768
hidden_dim: 768
out_dim: 768
Model loaded.
Embedding matrices:
x1: (9999, 768)
x2: (9999, 768)
Projected embeddings:
z1: (9999, 768)
z2: (9999, 768)
Output dataframe shape: (9999, 1539)
Saved projected embeddings to: projected_embeddings.parquet
